In [1]:
import json
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

DATA_DIR = "bpi_mca_dataset"

train_traces = json.load(open(f"{DATA_DIR}/X_train.json"))
test_traces  = json.load(open(f"{DATA_DIR}/X_test.json"))

df = pd.read_csv(f"{DATA_DIR}/y_test.csv")
test_labels = df["label"].values
injection_type = df["type"].values

In [2]:
act_counts = Counter(a for t in train_traces for a in t)
total = sum(act_counts.values())

P = {a: act_counts[a]/total for a in act_counts}

def semantic_score(trace):
    return np.mean([
        -np.log(P.get(a, 1e-8))
        for a in trace
    ])

In [3]:
def build_bigram(traces):
    counts = Counter()
    for t in traces:
        for i in range(len(t)-1):
            counts[(t[i], t[i+1])] += 1
    total = sum(counts.values())
    return {k: v/total for k,v in counts.items()}

baseline = build_bigram(train_traces)

def kl_score(trace, baseline):
    if len(trace) <= 1:
        return 0

    counts = Counter((trace[i], trace[i+1]) for i in range(len(trace)-1))
    total = sum(counts.values())

    kl = 0
    eps = 1e-8

    for pair, cnt in counts.items():
        p = cnt / total
        q = baseline.get(pair, eps)
        kl += p * np.log(p / q)

    return kl

In [4]:
from collections import defaultdict
import numpy as np

def build_order(traces):
    pos = defaultdict(list)
    
    for t in traces:
        for i, a in enumerate(t):
            pos[a].append(i / max(1, len(t)-1))
    
    return {a: np.mean(v) for a, v in pos.items()}

# Build once using training data
modal = build_order(train_traces)

In [5]:
def temporal_score(trace):
    pairs = 0
    violations = 0

    n = len(trace)

    for i in range(n):
        for j in range(i + 1, n):

            a = trace[i]
            b = trace[j]

            if a in modal and b in modal:
                pairs += 1

                if modal[a] > modal[b]:
                    violations += 1

    if pairs == 0:
        return 0

    ratio = violations / pairs

    # 🔥 amplify signal
    return ratio ** 0.5

In [6]:
S_scores = np.array([semantic_score(t) for t in test_traces])
C_scores = np.array([kl_score(t, baseline) for t in test_traces])


train_S = np.array([semantic_score(t) for t in train_traces])
train_C = np.array([kl_score(t, baseline) for t in train_traces])
T_scores = np.array([temporal_score(t) for t in test_traces])
train_T  = np.array([temporal_score(t) for t in train_traces])

In [7]:
sc_S = StandardScaler().fit(train_S.reshape(-1,1))
sc_C = StandardScaler().fit(train_C.reshape(-1,1))
sc_T = StandardScaler().fit(train_T.reshape(-1,1))

S = sc_S.transform(S_scores.reshape(-1,1)).flatten()
C = sc_C.transform(C_scores.reshape(-1,1)).flatten()
T = sc_T.transform(T_scores.reshape(-1,1)).flatten()

In [8]:
alpha, beta, gamma = 0.33, 0.34, 0.33

scores = alpha*S + beta*C + gamma*T

In [9]:
train_scores = (
    alpha * sc_S.transform(train_S.reshape(-1,1)).flatten() +
    beta  * sc_C.transform(train_C.reshape(-1,1)).flatten() +
    gamma * sc_T.transform(train_T.reshape(-1,1)).flatten()
)

threshold = np.percentile(train_scores, 70)
preds = (scores >= threshold).astype(int)

In [10]:
print(confusion_matrix(test_labels, preds))
print(classification_report(test_labels, preds))

auc = roc_auc_score(test_labels, scores)
print("ROC-AUC:", auc)

[[1477 1014]
 [ 982 6491]]
              precision    recall  f1-score   support

           0       0.60      0.59      0.60      2491
           1       0.86      0.87      0.87      7473

    accuracy                           0.80      9964
   macro avg       0.73      0.73      0.73      9964
weighted avg       0.80      0.80      0.80      9964

ROC-AUC: 0.8833310690599098


In [11]:
df_eval = pd.DataFrame({
    "type": injection_type,
    "S": S,
    "C": C,
    "T": T
})

print("\nSemantic anomalies:")
print(df_eval[df_eval["type"]=="S"][["S","C","T"]].mean())

print("\nControl-flow anomalies:")
print(df_eval[df_eval["type"]=="C"][["S","C","T"]].mean())

print("\nTemporal anomalies:")
print(df_eval[df_eval["type"]=="T"][["S","C","T"]].mean())


Semantic anomalies:
S    2.216692
C    7.861951
T    0.863665
dtype: float64

Control-flow anomalies:
S    -0.018213
C    11.139010
T     1.021023
dtype: float64

Temporal anomalies:
S   -0.018213
C    4.744511
T    0.331567
dtype: float64
